# 导入 (import)

In [ ]:
import pandas as pd

"""
处理 jsonl 为 pandas dataframe
Process jsonl to pandas dataframe
"""
words_df = pd.read_json("./crawler_output.jl", orient="records", lines=True)
words_df.head()

# 预处理

In [ ]:
words_df.rename(columns={"word": "spelling"}, inplace=True)
words_df = words_df.astype(
    {
        "spelling": "string",
        "part_of_speech": "string",
        "phonetic_symbol": object,
        "definitions": object,
    }
)

print(words_df.shape)
words_df.head()

# 展开音标

In [ ]:
phonetic_symbols = pd.DataFrame(
    words_df["phonetic_symbol"].tolist(),
    columns=["phonetic_bre", "phonetic_ame"],
)


words_df = pd.concat(
    [words_df.drop("phonetic_symbol", axis=1), phonetic_symbols], axis=1
)
words_df = words_df.astype({"phonetic_bre": "string", "phonetic_ame": "string"})

print(words_df.shape)
words_df.head()

# 展开释义

In [ ]:
assert (
    words_df["definitions"].isnull().sum() == 0
), "数据处理错误，存在缺失的 definitions！Data processing error, missing definitions!"

words_df = words_df.explode("definitions")
definitions = pd.json_normalize(words_df["definitions"])
definitions = definitions.rename(
    columns={"definition.en": "def_src", "definition.zh": "def_tgt"}
)
definitions = definitions.drop(columns=["cefr"])

words_df = pd.concat([words_df.drop(columns=["definitions"]), definitions], axis=1)
words_df = words_df.astype({"def_src": "string", "def_tgt": "string"})

assert len(definitions) == len(
    words_df
), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"

print(words_df.shape)
words_df.head()

# 展开例子

In [ ]:
words_df = words_df.explode("examples")

# 处理 examples 列，空的例子用 {} 填充，方便继续 json normalize 展开
words_df["examples"] = words_df["examples"].apply(
    lambda x: x if isinstance(x, dict) else {}
)
examples = pd.json_normalize(words_df["examples"])
examples = examples.rename(columns={"en": "ex_src", "zh": "ex_tgt"})

words_df = pd.concat([words_df.drop(columns=["examples"]), examples], axis=1)
words_df = words_df.astype({"ex_src": "string", "ex_tgt": "string"})
words_df["ex_src"] = words_df["ex_src"].fillna("")
words_df["ex_tgt"] = words_df["ex_tgt"].fillna("")

assert len(examples) == len(
    words_df
), "数据处理错误，行数不匹配！Data processing error, row count mismatch!"
print(words_df.shape)
words_df.head()

# 清洗（Cleaning）

In [ ]:
import re

def clean(input: str) -> str:
    # 去除多余的行
    # Remove unnecessary new lines
    input = re.sub(r"\s*\n", " ", input)
    
    # 去除没必要的空格
    # Remove unnecessary spaces
    result: str = ""
    str_list = re.split(r"\s+", input)
    str_list_len = len(str_list)
    for i, sub_str in enumerate(str_list):
        if i == str_list_len - 1:
            result += str_list[-1]
        else:
            result += sub_str + " "
    return result

def assert_is_not_null_or_empty(s: pd.Series) -> bool:
    """
    判断字符串是否是空或者者全是空白的字符串
    Determine if the string is empty or consists entirely of whitespace characters
    """
    return s.isnull().sum() > 0 or s == ""

def assert_not_blank_just_empty(s: pd.Series) -> None:
    """
    应该清洗后的空字符串不是空白，应该是空字符串
    """
    assert s.str.match(r"^\s+$").sum() == 0, "数据处理错误，存在空白的字符串！Data processing error, blank string exists!"


words_df = words_df.reset_index(drop=True)
words_df["spelling"] = words_df["spelling"].apply(clean)
words_df["phonetic_bre"] = words_df["phonetic_bre"].apply(clean)
words_df["phonetic_ame"] = words_df["phonetic_ame"].apply(clean)
words_df["part_of_speech"] = words_df["part_of_speech"].apply(clean)
words_df["def_src"] = words_df["def_src"].apply(clean)
words_df["def_tgt"] = words_df["def_tgt"].apply(clean)
words_df["ex_src"] = words_df["ex_src"].apply(clean)
words_df["ex_tgt"] = words_df["ex_tgt"].apply(clean)
words_df = words_df.drop_duplicates()

# spelling 不可能是空或者者全是空白的字符串
assert_not_blank_just_empty(words_df["phonetic_bre"] )
assert_not_blank_just_empty(words_df["phonetic_ame"] )
assert_not_blank_just_empty(words_df["part_of_speech"] )
assert_not_blank_just_empty(words_df["def_src"] )
assert_not_blank_just_empty(words_df["def_tgt"] )
assert_not_blank_just_empty(words_df["ex_src"] )
assert_not_blank_just_empty(words_df["ex_tgt"] )

# schema 约束检查
assert_is_not_null_or_empty(words_df["spelling"] )
assert_is_not_null_or_empty(words_df["part_of_speech"] )

print(words_df.shape)
words_df.head()

# 拆表为符合数据库定义的 schema 进行导出

In [ ]:
from uuid import uuid7
import time

unix_time = time.time_ns() // 1_000_000

def uuid7_series(n: int) -> pd.Series:
    return pd.Series([str(uuid7()) for _ in range(n)])

def assert_words_df_not_duplicated() -> bool:
    assert (
        words_df.duplicated().sum() == 0
    ), "数据处理错误，存在重复的行！Data processing error, duplicated rows exist!"

def str_is_empty(s: pd.Series) -> pd.Series:
    return s.isnull() | (s == "")

## words 表

In [ ]:
words_tb = words_df[["spelling", "phonetic_bre", "phonetic_ame"]]
words_tb = words_tb.drop_duplicates(subset=["spelling"]).reset_index(drop=True)

words_tb.insert(loc=0, column="word_id", value=uuid7_series(len(words_tb)))
words_tb["created_at"] = unix_time
words_tb["updated_at"] = unix_time

# 注意后面还得生成一个指纹
#words_tb.to_json(
#    "./words.jsonl", orient="records", lines=True, force_ascii=False
#)

print(words_tb.shape)
words_tb.head()

In [ ]:
# 回填
words_df = words_df.merge(
    words_tb[["spelling", "word_id"]],
    on="spelling",
    how="left",
    validate="many_to_one",
)

assert_not_blank_just_empty(words_df["word_id"])
assert_words_df_not_duplicated()

print(words_df.shape)
words_df.head()

## word_poses 表

In [ ]:
word_poses_tb = words_df[["word_id", "part_of_speech"]]
word_poses_tb = word_poses_tb.drop_duplicates(subset=["word_id", "part_of_speech"])

word_poses_tb.insert(
    loc=0,
    column="pose_id",
    value=uuid7_series(len(word_poses_tb))
)

word_poses_tb["created_at"] = unix_time
word_poses_tb["updated_at"] = unix_time

word_poses_tb.to_json(
    "./word_poses.jsonl", orient="records", lines=True, force_ascii=False
)
print(word_poses_tb.shape)
word_poses_tb.head()

In [ ]:
# 回填
words_df = words_df.merge(
    word_poses_tb[["word_id", "part_of_speech", "pose_id"]],
    on=["word_id", "part_of_speech"],
    how="left",
    validate="many_to_one",
)
assert_not_blank_just_empty(words_df["pose_id"])

print(words_df.shape)
words_df.head()

## definitions 表

In [ ]:
definitions_tb = words_df[["pose_id", "def_src", "def_tgt"]].drop_duplicates()
definitions_tb = definitions_tb.rename(columns={"pose_id": "word_pos_id"})

definitions_tb.insert(loc=0, column="def_id", value=uuid7_series(len(definitions_tb)))
definitions_tb["created_at"] = unix_time
definitions_tb["updated_at"] = unix_time

definitions_tb.to_json(
    "./definitions.jsonl", orient="records", lines=True, force_ascii=False
)

print(definitions_tb.shape)
definitions_tb.head()

In [ ]:
# 回填
words_df = words_df.merge(
    definitions_tb[["def_id", "word_pos_id", "def_src", "def_tgt"]],
    left_on=["pose_id", "def_src", "def_tgt"],
    right_on=["word_pos_id", "def_src", "def_tgt"],
    how="left",
    validate="many_to_one",
)

# 已经有 pose_id 了，这个是 join 临时用的
words_df = words_df.drop(columns=["word_pos_id"])

assert_not_blank_just_empty(words_df["def_id"])
assert_words_df_not_duplicated()

print(words_df.shape)
words_df.head()

## examples 表

In [ ]:
examples_tb = words_df[["def_id", "ex_src", "ex_tgt"]].drop_duplicates()

# drop 掉同时没有中英文例句的行
examples_tb = examples_tb[
    ~(str_is_empty(examples_tb["ex_src"]) & str_is_empty(examples_tb["ex_tgt"]))
]

assert (
    examples_tb[
        (str_is_empty(examples_tb["ex_src"]) & str_is_empty(examples_tb["ex_tgt"]))
    ].shape[0]
    == 0
), "数据处理错误，存在缺失的 ex_src 或 ex_tgt！Data processing error, missing ex_src or ex_tgt!"

examples_tb.insert(loc=0, column="exp_id", value=uuid7_series(len(examples_tb)))
examples_tb["created_at"] = unix_time
examples_tb["updated_at"] = unix_time
assert_not_blank_just_empty(examples_tb["def_id"])

examples_tb.to_json(
    "./examples.jsonl", orient="records", lines=True, force_ascii=False
)

print(examples_tb.shape)
examples_tb.head()

## 指纹

In [ ]:
import hashlib
import json


FINGERPRINT_VERSION = "v1"

def fingerprint_part(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value)

def make_fingerprint(namespace: str, *parts: object) -> str:
    payload = json.dumps(
        [
            FINGERPRINT_VERSION,
            namespace,
            *[fingerprint_part(part) for part in parts],
        ],
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def sort_fingerprints(fingerprints: list[str]) -> list[str]:
    return sorted(fingerprint_part(fingerprint) for fingerprint in fingerprints)

def make_word_child_content_fingerprint(
    part_of_speech: str,
    def_src: str,
    def_tgt: str,
    ex_src: str,
    ex_tgt: str,
 ) -> str:
    return make_fingerprint(
        "word_child_content",
        part_of_speech,
        def_src,
        def_tgt,
        ex_src,
        ex_tgt,
    )

def make_word_fingerprint(
    spelling: str,
    phonetic_bre: str,
    phonetic_ame: str,
    child_content_fingerprints: list[str],
 ) -> str:
    return make_fingerprint(
        "word",
        spelling,
        phonetic_bre,
        phonetic_ame,
        *child_content_fingerprints,
    )

## words 表导出
因为添加指纹方便后续同步，而根据要输入的东西生成的指纹

In [ ]:
words_df["word_child_content_fingerprint"] = words_df.apply(
    lambda row: make_word_child_content_fingerprint(
        row["part_of_speech"],
        row["def_src"],
        row["def_tgt"],
        row["ex_src"],
        row["ex_tgt"],
    ),
    axis=1,
 )

# 记得排序防止顺序影响不同产生不同的 fingerprints
word_child_content_fingerprints = (
    words_df.groupby("word_id")["word_child_content_fingerprint"]
    .agg(lambda fingerprints: sort_fingerprints(fingerprints.tolist()))
    .to_dict()
)

words_tb.insert(
    loc=2,
    column="fingerprint",
    value=words_tb.apply(
        lambda row: make_word_fingerprint(
            row["spelling"],
            row["phonetic_bre"],
            row["phonetic_ame"],
            word_child_content_fingerprints.get(row["word_id"], []),
        ),
        axis=1,
    ),
)

assert (
    words_tb["fingerprint"].duplicated().sum() == 0
), "数据处理错误，存在重复的 word fingerprint！Data processing error, duplicated word fingerprints exist!"

words_tb.to_json(
    "./words.jsonl", orient="records", lines=True, force_ascii=False
)

print(words_tb.shape)
words_tb.head()